In [1]:
import sys
print(f"✅ Python 3.10 + JupyterLab = PERFECT!")
import ipywidgets as widgets
btn = widgets.Button(description="🎉 READY!")
btn

✅ Python 3.10 + JupyterLab = PERFECT!


Button(description='🎉 READY!', style=ButtonStyle())

In [2]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime
import hashlib

# Database
conn = sqlite3.connect('tradeflow.db')
conn.execute('''CREATE TABLE IF NOT EXISTS users (username TEXT PRIMARY KEY, password_hash TEXT, role TEXT, store TEXT)''')
conn.execute('''CREATE TABLE IF NOT EXISTS sales (id INTEGER PRIMARY KEY, timestamp TEXT, username TEXT, store TEXT, product TEXT, quantity INTEGER, price REAL, profit REAL)''')

# Users: admin/123456 | managerNYC/123456
users = [
    ('admin', '8d969eef6ecad3c29a3a629280e', 'admin', None),
    ('managerNYC', '8d969eef6ecad3c29a3a629280e', 'manager', 'NYC')
]
conn.executemany("INSERT OR IGNORE INTO users VALUES (?,?,?,?)", users)
conn.commit()

user_session = {}
print("✅ DATABASE READY!")
print("👥 admin/123456 | managerNYC/123456")

✅ DATABASE READY!
👥 admin/123456 | managerNYC/123456


In [3]:
# DEBUG: Check database
conn = sqlite3.connect('tradeflow.db', check_same_thread=False)
df_users = pd.read_sql_query("SELECT * FROM users", conn)
print("ALL USERS IN DB:")
print(df_users)
print("\nHimadri hash check:")
phash = hashlib.sha256("29092005".encode()).hexdigest()[:15]
print(f"Password hash: {phash}")

ALL USERS IN DB:
     username                password_hash     role store
0     Himadri              1a122742b87e933    admin  None
1       admin              8d969eef6ecad3c    admin  None
2  managerNYC  8d969eef6ecad3c29a3a629280e  manager   NYC
3        Diya              0c8b4777b53dcb8    admin  None

Himadri hash check:
Password hash: 7cd3b0a8a11192a


In [4]:
# CHANGE PASSWORD
TARGET_USER = "Himadri"        # ← Existing user
NEW_PASSWORD = "KIIT-24"    # ← New password

conn = sqlite3.connect('tradeflow.db', check_same_thread=False)
phash = hashlib.sha256(NEW_PASSWORD.encode()).hexdigest()[:15]

conn.execute("UPDATE users SET password_hash=? WHERE username=?", 
             (phash, TARGET_USER))
conn.commit()

print(f"✅ {TARGET_USER} password changed to: {NEW_PASSWORD}")
print(f"New hash: {phash}")

✅ Himadri password changed to: KIIT-24
New hash: 1a122742b87e933


In [5]:
# ADD NEW USER - CHANGE THESE 2 LINES
NEW_USERNAME = "Diya"      # ← YOUR USERNAME
NEW_PASSWORD = "1B-24"   # ← YOUR PASSWORD

conn = sqlite3.connect('tradeflow.db', check_same_thread=False)
phash = hashlib.sha256(NEW_PASSWORD.encode()).hexdigest()[:15]

conn.execute("INSERT OR REPLACE INTO users VALUES (?,?,?,?)", 
             (NEW_USERNAME, phash, 'admin', None))
conn.commit()

print(f"✅ NEW USER ADDED: {NEW_USERNAME}/{NEW_PASSWORD}")
print(f"Hash: {phash}")

✅ NEW USER ADDED: Diya/1B-24
Hash: 0c8b4777b53dcb8


In [ ]:
import threading
thread_local = threading.local()

def get_conn():
    if not hasattr(thread_local, 'conn'):
        thread_local.conn = sqlite3.connect('tradeflow.db', check_same_thread=False)
    return thread_local.conn

login_out = widgets.Output()

def login(username, password):
    conn = get_conn()
    # FIXED: Use FULL 15-char hash match
    phash = hashlib.sha256(password.encode()).hexdigest()[:15]
    df = pd.read_sql_query("SELECT * FROM users WHERE username=? AND password_hash=?", 
                          conn, params=(username, phash))
    if len(df) > 0:
        user_session.update({
            'logged_in': True, 
            'username': username, 
            'role': df.iloc[0]['role'], 
            'store': df.iloc[0]['store']
        })
        return True
    return False

# Login Form
login_user = widgets.Text(value='Himadri', description='Username:')
login_pass = widgets.Password(value='29092005', description='Password:')
login_btn = widgets.Button(description="🔐 LOGIN", button_style='primary')

def on_login(b):
    with login_out:
        clear_output()
        username = login_user.value.strip()
        password = login_pass.value
        print(f"DEBUG: Trying {username}/{password}")
        print(f"Hash: {hashlib.sha256(password.encode()).hexdigest()[:15]}")
        
        if login(username, password):
            print(f"✅ SUCCESS: {user_session['username']} logged in!")
        else:
            print("❌ FAILED - Check spelling/case")

login_btn.on_click(on_login)

display(widgets.VBox([
    widgets.HTML("<h2>🔐 Login (Debug Mode)</h2>"),
    login_user, login_pass, login_btn, login_out
]))

In [7]:
dash_out = widgets.Output()

def get_kpis():
    df = pd.read_sql_query("SELECT * FROM sales", conn)
    return {'revenue': df['price'].sum() if len(df)>0 else 0, 'profit': df['profit'].sum() if len(df)>0 else 0}

def show_dash():
    if not user_session.get('logged_in'): 
        with dash_out: print("⚠️ LOGIN FIRST")
        return
    with dash_out:
        clear_output()
        kpis = get_kpis()
        display(HTML(f"""
        <h1>🏪 TradeFlow Pro - {user_session['username']}</h1>
        <div style='display:flex;gap:20px'>
          <div style='background:#4ecdc4;padding:20px;border-radius:10px;color:white'>
            <h2>${kpis['revenue']:,.0f}</h2><p>💰 Revenue</p>
          </div>
          <div style='background:#45b7d1;padding:20px;border-radius:10px;color:white'>
            <h2>${kpis['profit']:,.0f}</h2><p>💵 Profit</p>
          </div>
        </div>
        """))

show_dash()
display(dash_out)

Output()

In [8]:
sales_out = widgets.Output()

def add_sale(username, store, product, qty, price):
    conn = sqlite3.connect('tradeflow.db', check_same_thread=False)
    profit = (price * 0.25) * qty  # 25% margin
    conn.execute("""INSERT INTO sales (timestamp, username, store, product, quantity, price, profit) 
                    VALUES (datetime('now'), ?,?,?,?, ?, ?)""", 
                 (username, store, product, qty, price, profit))
    conn.commit()

if user_session.get('role') != 'viewer':
    # Role-based stores
    stores = ['NYC','LA','Chicago','Miami','Boston'] if user_session['role']=='admin' else [user_session['store']]
    
    store_w = widgets.Dropdown(options=stores, value=stores[0], description='Store:')
    prod_w = widgets.Dropdown(options=['iPhone 15','MacBook','Samsung','Laptop','Headphones'], description='Product:')
    qty_w = widgets.IntSlider(value=1, min=1, max=20, description='Quantity:')
    price_w = widgets.FloatSlider(value=999, min=100, max=2500, description='Price $:')
    sale_btn = widgets.Button(description="🚀 ADD SALE", button_style='success')

    def on_sale(b):
        add_sale(user_session['username'], store_w.value, prod_w.value, 
                qty_w.value, price_w.value)
        with sales_out:
            clear_output()
            print(f"✅ SALE ADDED!")
            print(f"{qty_w.value}x {prod_w.value} @ ${price_w.value}")
            print("🔄 Dashboard updating...")
            show_dash()  # Refresh KPIs

    sale_btn.on_click(on_sale)
    
    display(widgets.VBox([
        widgets.HTML("<h2>➕ Add Sale (Live Update)</h2>"),
        store_w, prod_w, 
        widgets.HBox([qty_w, price_w]),
        sale_btn, sales_out
    ]))
else:
    print("👁️ Viewer: Cannot add sales")

In [9]:
import plotly.express as px  

charts_out = widgets.Output()

def update_charts():
    with charts_out:
        clear_output()
        conn = sqlite3.connect('tradeflow.db', check_same_thread=False)
        df = pd.read_sql_query("SELECT * FROM sales ORDER BY timestamp DESC LIMIT 100", conn)
        
        if len(df) == 0:
            print("📊 No sales yet!")
            return
        
        # Revenue by Store
        store_rev = df.groupby('store')['price'].sum().sort_values(ascending=False)
        fig1 = px.bar(x=store_rev.index, y=store_rev.values, 
                     title="💰 Revenue by Store (LIVE)")
        
        # Profit by Product  
        prod_profit = df.groupby('product')['profit'].sum().sort_values(ascending=False)
        fig2 = px.pie(values=prod_profit.values, names=prod_profit.index, 
                     title="📈 Profit by Product")
        
        # Recent sales
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        recent = df[['timestamp','store','product','quantity','profit']].head(10)
        recent['profit'] = recent['profit'].round(2)
        
        display(fig1, fig2)
        display(recent)

update_charts()
display(charts_out)

Output()

In [10]:
edit_out = widgets.Output()

def edit_sale(sale_id, new_qty, new_price):
    conn = sqlite3.connect('tradeflow.db', check_same_thread=False)
    profit = (new_price * 0.25) * new_qty
    conn.execute("UPDATE sales SET quantity=?, price=?, profit=? WHERE id=?", 
                (new_qty, new_price, profit, sale_id))
    conn.commit()
    print(f"✅ Sale ID {sale_id} updated: {new_qty}x${new_price}")

# Show recent sales
conn = sqlite3.connect('tradeflow.db', check_same_thread=False)
df_sales = pd.read_sql_query("SELECT * FROM sales ORDER BY id DESC LIMIT 10", conn)
print("📋 Recent Sales (Copy ID to edit):")
display(df_sales[['id','store','product','quantity','price']])

# Edit form
sale_id_w = widgets.Dropdown(options=df_sales['id'].tolist(), description='Sale ID:')
qty_w = widgets.IntSlider(value=1, min=1, max=50, description='New Qty:')
price_w = widgets.FloatSlider(value=1000, min=50, max=5000, description='New Price:')
edit_btn = widgets.Button(description="✏️ UPDATE SALE", button_style='warning')  # FIXED

def on_edit(b):
    edit_sale(sale_id_w.value, qty_w.value, price_w.value)
    with edit_out:
        clear_output()
        print("✅ UPDATED! Refresh dashboard/charts")
        show_dash()

edit_btn.on_click(on_edit)
display(widgets.VBox([sale_id_w, qty_w, price_w, edit_btn, edit_out]))

📋 Recent Sales (Copy ID to edit):


,id,store,product,quantity,price
0,11,Boston,MacBook,5,3200.64
1,10,LA,Headphones,15,1495.60
2,9,LA,Laptop,7,1538.80
3,8,LA,Samsung,10,1841.00
4,7,LA,MacBook,8,1581.90
5,6,LA,iPhone 15,10,2035.30
6,5,Chicago,iPhone 15,8,2035.30
7,4,Chicago,Headphones,10,2644.80
8,3,Chicago,Samsung,4,1063.80
9,2,Chicago,MacBook,7,1819.40


In [11]:
# LIVE AUTO-UPDATE (Refreshes every 10s)
import time

def auto_refresh():
    for i in range(20):  # 3+ minutes demo
        time.sleep(10)
        print(f"🔄 Auto-refresh #{i+1}")
        show_dash()
        update_charts()

# Start background refresh
refresh_thread = threading.Thread(target=auto_refresh, daemon=True)
refresh_thread.start()
print("🚀 LIVE AUTO-REFRESH STARTED!")

🚀 LIVE AUTO-REFRESH STARTED!


In [ ]:
def export_pnl():
    conn = sqlite3.connect('tradeflow.db', check_same_thread=False)
    df = pd.read_sql_query("SELECT * FROM sales", conn)
    if len(df) == 0:
        print("No sales to export")
        return
    
    filename = f"TradeFlow_PnL_{user_session['username']}_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
    df.to_csv(filename, index=False)
    print(f"✅ P&L EXPORTED: {filename}")
    print(f"Revenue: ${df['price'].sum():,.0f} | Profit: ${df['profit'].sum():,.0f}")

export_btn = widgets.Button(description="📊 Export P&L Report", button_style='success')
export_btn.on_click(lambda b: export_pnl())
display(export_btn)

Button(button_style='success', description='📊 Export P&L Report', style=ButtonStyle())

🔄 Auto-refresh #1


🔄 Auto-refresh #2


🔄 Auto-refresh #3


🔄 Auto-refresh #4
